In [1]:
import os
import tensorflow as tf

import numpy as np
import pandas as pd

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.vgg16 import preprocess_input

from tensorflow.keras.optimizers import Adam
import tensorflow as tf

import requests
from PIL import Image
from io import BytesIO



In [2]:
seed_value = 42  # You can choose your own seed value
os.environ['PYTHONHASHSEED'] = str(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [3]:
df = pd.read_pickle("electronics_image.pkl")
df

,reviewerID,asin,summary,reviewText,image,vote,overall
0,A7362LXMQEM6W,0511189877,Paid full price,Worked out of the box with my cable box in nor...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
1,A198CYZEQVCDBV,0594033926,Nice but Not for Me,It's glossier than I thought it was going to b...,https://images-na.ssl-images-amazon.com/images...,5,4.0
2,A21WU4E46K96UU,0594459451,Original equipment,"Works like a dream, now my original nook is al...",https://images-na.ssl-images-amazon.com/images...,NaN,5.0
3,A2GZKMU3019XO3,059449771X,"Charged Nook, happy life",A++++ Don't waste your money on any other cord...,https://images-na.ssl-images-amazon.com/images...,3,5.0
4,A2AZSLQBRA8PJ7,073530498X,Photo album fun,Happy with my purchase. I'm confused by some o...,https://images-na.ssl-images-amazon.com/images...,178,5.0
...,...,...,...,...,...,...,...
346057,ASRZM1V4XKHT9,B01HJ6YR5G,I love this little sticker,I love this little sticker! I bought it for my...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346058,ABK76HNPD2O1K,B01HJDNL60,Garbage!,Stay away... I don't know how dashcams can be ...,https://images-na.ssl-images-amazon.com/images...,NaN,1.0
346059,AX2XWOFFJARGH,B01HJFXET2,Worth!,If you have more than day 5000 dollars of gear...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346060,A120TWL6P7XFYC,B01HJFUYFY,Go pro Quality without the go pro price,I have been using this camera for about 5 mont...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0


In [4]:
df.dropna(subset=['reviewText'], inplace = True)
df

,reviewerID,asin,summary,reviewText,image,vote,overall
0,A7362LXMQEM6W,0511189877,Paid full price,Worked out of the box with my cable box in nor...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
1,A198CYZEQVCDBV,0594033926,Nice but Not for Me,It's glossier than I thought it was going to b...,https://images-na.ssl-images-amazon.com/images...,5,4.0
2,A21WU4E46K96UU,0594459451,Original equipment,"Works like a dream, now my original nook is al...",https://images-na.ssl-images-amazon.com/images...,NaN,5.0
3,A2GZKMU3019XO3,059449771X,"Charged Nook, happy life",A++++ Don't waste your money on any other cord...,https://images-na.ssl-images-amazon.com/images...,3,5.0
4,A2AZSLQBRA8PJ7,073530498X,Photo album fun,Happy with my purchase. I'm confused by some o...,https://images-na.ssl-images-amazon.com/images...,178,5.0
...,...,...,...,...,...,...,...
346057,ASRZM1V4XKHT9,B01HJ6YR5G,I love this little sticker,I love this little sticker! I bought it for my...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346058,ABK76HNPD2O1K,B01HJDNL60,Garbage!,Stay away... I don't know how dashcams can be ...,https://images-na.ssl-images-amazon.com/images...,NaN,1.0
346059,AX2XWOFFJARGH,B01HJFXET2,Worth!,If you have more than day 5000 dollars of gear...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346060,A120TWL6P7XFYC,B01HJFUYFY,Go pro Quality without the go pro price,I have been using this camera for about 5 mont...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0


In [5]:
df = df.reset_index(drop=True)
df 

,reviewerID,asin,summary,reviewText,image,vote,overall
0,A7362LXMQEM6W,0511189877,Paid full price,Worked out of the box with my cable box in nor...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
1,A198CYZEQVCDBV,0594033926,Nice but Not for Me,It's glossier than I thought it was going to b...,https://images-na.ssl-images-amazon.com/images...,5,4.0
2,A21WU4E46K96UU,0594459451,Original equipment,"Works like a dream, now my original nook is al...",https://images-na.ssl-images-amazon.com/images...,NaN,5.0
3,A2GZKMU3019XO3,059449771X,"Charged Nook, happy life",A++++ Don't waste your money on any other cord...,https://images-na.ssl-images-amazon.com/images...,3,5.0
4,A2AZSLQBRA8PJ7,073530498X,Photo album fun,Happy with my purchase. I'm confused by some o...,https://images-na.ssl-images-amazon.com/images...,178,5.0
...,...,...,...,...,...,...,...
346057,ASRZM1V4XKHT9,B01HJ6YR5G,I love this little sticker,I love this little sticker! I bought it for my...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346058,ABK76HNPD2O1K,B01HJDNL60,Garbage!,Stay away... I don't know how dashcams can be ...,https://images-na.ssl-images-amazon.com/images...,NaN,1.0
346059,AX2XWOFFJARGH,B01HJFXET2,Worth!,If you have more than day 5000 dollars of gear...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0
346060,A120TWL6P7XFYC,B01HJFUYFY,Go pro Quality without the go pro price,I have been using this camera for about 5 mont...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0


In [6]:
df['image'] = df['image'].astype(str)
df['image'] = df['image'].str.strip("[']'")

In [7]:
from skimage import io

In [8]:
from tqdm.auto import tqdm 
from tqdm import tqdm_pandas 

c:\Users\LIXINZHE\anaconda3\envs\GPU\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def check_img(x):
    try:
        imgs = io.imread(x)
        return x
    except:
        return None

In [14]:
tqdm.pandas()

In [15]:
new_cols = df.loc[:, 'image'].progress_map(check_img)

  0%|          | 0/346062 [00:00<?, ?it/s]

  0%|          | 407/346062 [02:37<51:39:26,  1.86it/s]

In [12]:
df.image[0]

'https://images-na.ssl-images-amazon.com/images/I/81XYkHdfrjL._SY88.jpg'

In [10]:
for img in df.image.values():
    response = requests.get(img)

0         https://images-na.ssl-images-amazon.com/images...
1         https://images-na.ssl-images-amazon.com/images...
2         https://images-na.ssl-images-amazon.com/images...
3         https://images-na.ssl-images-amazon.com/images...
4         https://images-na.ssl-images-amazon.com/images...
                                ...                        
346057    https://images-na.ssl-images-amazon.com/images...
346058    https://images-na.ssl-images-amazon.com/images...
346059    https://images-na.ssl-images-amazon.com/images...
346060    https://images-na.ssl-images-amazon.com/images...
346061    https://images-na.ssl-images-amazon.com/images...
Name: image, Length: 346062, dtype: object

In [8]:
df1 = df[:500000]
df2 = df[500000:100000] 
df3 = df[100000:150000]
df4 = df[150000:200000]
df5 = df[200000:250000]
df6 = df[250000:300000]
df7 = df[300000:]

In [11]:
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2L(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='avg')
x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

j = 1
def process_image(url):
    # link_list = []
    try:
        response = requests.get(url)
        img = Image.open(BytesIO(response.content)).resize((224, 224))
        img = img.convert("RGB")
        # link_list.append(url)
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)

    except Exception as e:
        # Create a zero array for failed image fetches
        print(f"Error fetching image: {e}")
        img_array = np.zeros((224, 224, 3))
        img_array = preprocess_input(img_array)

    # if link_list == []:
    #     return None 
    # else:
    #     return ','.join(link_list)

    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    return prediction

# Function to process a row of the DataFrame
def process_images(urls):

    global j
    print("Processed {%i} images" %j)
    j+=1
    if urls == 'nan':
        return process_image(None).flatten()
    else:
        urls = urls.split(("', '"))
        predictions = np.array([process_image(url) for url in urls])
        avg_prediction = np.mean(predictions, axis=0).squeeze()
        return avg_prediction

# Apply the function to each row
sample_df['EfficientNetV2L'] = sample_df['image'].apply(process_images)

Processed {1} images
1/1 [==============================] - 7s 7s/step
Processed {2} images
1/1 [==============================] - 0s 37ms/step
Processed {3} images
1/1 [==============================] - 0s 33ms/step
Processed {4} images
1/1 [==============================] - 0s 38ms/step
Processed {5} images
1/1 [==============================] - 0s 51ms/step
Processed {6} images
1/1 [==============================] - 0s 48ms/step
Processed {7} images
1/1 [==============================] - 0s 55ms/step
Processed {8} images
1/1 [==============================] - 0s 47ms/step
Processed {9} images
1/1 [==============================] - 0s 56ms/step
Processed {10} images
1/1 [==============================] - 0s 64ms/step


C:\Users\LIXINZHE\AppData\Local\Temp\ipykernel_1488\800944465.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df['VGG16'] = sample_df['image'].apply(process_images)


In [9]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='max')
x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

img_array = np.zeros((224, 224, 3))

img_array = np.expand_dims(img_array, axis=0)
prediction = model.predict(img_array)

1/1 [==============================] - 4s 4s/step


In [19]:
prediction

In [10]:
prediction.flatten()

array([0.3002223 , 0.        , 0.        , ..., 0.        , 0.        ,
       0.01306493], dtype=float32)

In [2]:
x = pd.read_pickle('electronics_final.pkl')
x

,reviewerID,asin,summary,reviewText,image,vote,overall,VGG16,user,item,rating,text,bert,roberta
0,A7362LXMQEM6W,0511189877,Paid full price,Worked out of the box with my cable box in nor...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0,"[0.0, 2.104456, 17.229027, 30.3031, 0.0, 6.747...",207916,0,5.0,Worked out of the box with my cable box in nor...,"[-0.155445858836174, -0.43212705850601196, 0.1...","[-0.09964974224567413, 0.13190370798110962, -0..."
1,A198CYZEQVCDBV,0594033926,Nice but Not for Me,It's glossier than I thought it was going to b...,https://images-na.ssl-images-amazon.com/images...,5,4.0,"[2.3792214, 0.25898618, 12.55817, 13.3320465, ...",17954,1,4.0,It's glossier than I thought it was going to b...,"[-0.4509936273097992, -0.31166020035743713, 0....","[-0.08462080359458923, 0.14590221643447876, -0..."
2,A21WU4E46K96UU,0594459451,Original equipment,"Works like a dream, now my original nook is al...",https://images-na.ssl-images-amazon.com/images...,NaN,5.0,"[0.844841, 25.230316, 11.992286, 1.6452465, 0....",73526,2,5.0,"Works like a dream, now my original nook is al...","[0.0010838871821761131, -0.02658570371568203, ...","[-0.07300610095262527, 0.07474423944950104, -0..."
3,A2GZKMU3019XO3,059449771X,"Charged Nook, happy life",A++++ Don't waste your money on any other cord...,https://images-na.ssl-images-amazon.com/images...,3,5.0,"[0.0, 3.0880618, 22.295147, 0.0, 0.0, 0.0, 15....",102666,4,5.0,A++++ Don't waste your money on any other cord...,"[0.19765156507492065, -0.1666114330291748, 0.4...","[-0.07438350468873978, 0.10516618192195892, -0..."
4,A2AZSLQBRA8PJ7,073530498X,Photo album fun,Happy with my purchase. I'm confused by some o...,https://images-na.ssl-images-amazon.com/images...,178,5.0,"[0.0, 1.8128443, 7.178225, 13.095459, 0.0, 2.8...",91264,8,5.0,Happy with my purchase. I'm confused by some o...,"[-0.616042971611023, 0.03891818970441818, 0.53...","[-0.06578152626752853, 0.11396472901105881, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346057,ASRZM1V4XKHT9,B01HJ6YR5G,I love this little sticker,I love this little sticker! I bought it for my...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0,"[1.7661875, 0.0, 6.9169197, 0.0, 2.9944065, 2....",250259,102587,5.0,I love this little sticker! I bought it for my...,"[-0.29054608941078186, -0.18163947761058807, 0...","[-0.09928352385759354, 0.13887163996696472, -0..."
346058,ABK76HNPD2O1K,B01HJDNL60,Garbage!,Stay away... I don't know how dashcams can be ...,https://images-na.ssl-images-amazon.com/images...,NaN,1.0,"[0.0, 0.0, 15.375481, 0.0, 25.947186, 8.603322...",216450,102596,1.0,Stay away... I don't know how dashcams can be ...,"[-0.20375046133995056, -0.16710546612739563, 0...","[-0.07446134835481644, 0.0652696043252945, -0...."
346059,AX2XWOFFJARGH,B01HJFXET2,Worth!,If you have more than day 5000 dollars of gear...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0,"[2.0768476, 0.0, 13.468112, 0.5345977, 6.84269...",258508,102601,5.0,If you have more than day 5000 dollars of gear...,"[-0.39788004755973816, -0.177546426653862, 0.8...","[-0.04629392549395561, 0.10395243018865585, -0..."
346060,A120TWL6P7XFYC,B01HJFUYFY,Go pro Quality without the go pro price,I have been using this camera for about 5 mont...,https://images-na.ssl-images-amazon.com/images...,NaN,5.0,"[4.010207, 0.0, 0.0, 0.0, 9.330822, 0.0, 0.0, ...",3863,102600,5.0,I have been using this camera for about 5 mont...,"[0.2510935068130493, -0.18181343376636505, 0.5...","[-0.03520766273140907, 0.09927097707986832, -0..."


In [14]:
np.shape(x['VGG16'][0])

(4096,)

In [15]:
np.shape(prediction.flatten())

(4096,)

In [34]:
x['VGG16'].drop_duplicates()

In [4]:
x['VGG16'].apply(lambda x: x.tolist()).nunique()

TypeError: unhashable type: 'list'

In [9]:
j = 0
# Load VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='max')
x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

def process_image(url):
    try:
        response = requests.get(url)
        img = Image.open(BytesIO(response.content)).resize((224, 224))
        img = img.convert("RGB")
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)

    except Exception as e:
        # Create a zero array for failed image fetches
        print(f"Error fetching image: {e}")
        img_array = np.zeros((224, 224, 3))
        img_array = preprocess_input(img_array)

    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)
    return prediction

# Function to process a row of the DataFrame
def process_images(urls):

    global j
    print("Processed {%i} images" %j)
    j+=1
    if urls == 'nan':
        return process_image(None).flatten()
    else:
        urls = urls.split(("', '"))
        predictions = np.array([process_image(url) for url in urls])
        avg_prediction = np.mean(predictions, axis=0).squeeze()
        return avg_prediction

# Apply the function to each row
df4_1['VGG16'] = df4_1['image'].apply(process_images)




58889256/58889256 [==============================] - 6s 0us/step
Processed {0} images
1/1 [==============================] - 0s 75ms/step
Processed {1} images
1/1 [==============================] - 0s 82ms/step
Processed {2} images
1/1 [==============================] - 0s 84ms/step
Processed {3} images
1/1 [==============================] - 0s 77ms/step
Processed {4} images
1/1 [==============================] - 0s 89ms/step
Processed {5} images
1/1 [==============================] - 0s 77ms/step
Processed {6} images
1/1 [==============================] - 0s 81ms/step
Processed {7} images
1/1 [==============================] - 0s 76ms/step
Processed {8} images
1/1 [==============================] - 0s 80ms/step
Processed {9} images
1/1 [==============================] - 0s 77ms/step
Processed {10} images
1/1 [==============================] - 0s 77ms/step
Processed {11} images
1/1 [==============================] - 0s 73ms/step
Processed {12} images
1/1 [============================

In [ ]:
df4_1.to_pickle('VGG_Electronics_4_1.pkl')

,reviewerID,asin,summary,reviewText,image,vote,overall,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,processed_images
0,AJ9G2SE5KJPA8,B01EUXO8GY,Terrible fit on Gibson Les Paul,properly fit gibson le paul classic,https://images-na.ssl-images-amazon.com/images...,NaN,1,0.17895,0.02778,0.02790,0.02782,0.70974,0.02781,"[2.0750626e-10, 1.270388e-11, 7.912487e-09, 5...."
1,AL88R9LX7SOKB,B006MPUTBC,"Well-made, Quiet Cable.",buy cable base review plug marshall amp sure e...,nan,NaN,5,0.00646,0.09603,0.00646,0.70629,0.00646,0.17829,"[0.00045613412, 0.0019374342, 0.0007944805, 0...."
2,A23ZSXJ9HOFZNY,B003ZUEQLG,Perfect,love stand buy two perfect allow leave end pin...,https://images-na.ssl-images-amazon.com/images...,12,5,0.00993,0.15138,0.41389,0.00985,0.24574,0.16922,"[2.9707504e-15, 5.425265e-09, 6.995573e-14, 3...."
3,A1PEE2DR4Z0HSL,B0002M6B2M,"These strings are great, and cheap",string great cheap aware cheaper buy pack sepa...,nan,NaN,5,0.11135,0.58857,0.01301,0.01303,0.26113,0.01290,"[0.00045613412, 0.0019374342, 0.0007944805, 0...."
4,A1H2D403IZ5E6K,B0002MSZ3A,About the same as others.,several tune hammer pay anywhere handle practi...,nan,3,3,0.00381,0.07093,0.36546,0.00381,0.26050,0.29549,"[0.00045613412, 0.0019374342, 0.0007944805, 0...."
